# 4c. Cohort and retention analysis

**What an investor reads into this.** Cohorts show whether the customers being won today behave like the customers won in the past. A SaaS business can grow its headline numbers while each new group of customers leaves faster than the last; that shows up here long before it shows up in total revenue, because older, loyal customers mask it. The comparison has to be made at the same tenure: cumulative churn to date is misleading because an older cohort has simply had longer to lose customers, so it will always look worse even if it is the better cohort.

Definitions (`docs/metric_definitions.md`): cohort = signup quarter; month 0 = signup month; logo retention from subscription start and end dates (so cohorts back to 2017 are covered); revenue retention = cohort MRR relative to month 3, in-window cohorts only (2022Q1 onwards). A cohort cell is shown only if every customer in the cohort had reached that tenure by December 2025.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nb_utils import connect, q, sql_file, save_table
import charts as ch

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
con = connect()
END = pd.Timestamp("2025-12-01")
act = sql_file(con, "cohort_logo_activity")
act["month"] = pd.to_datetime(act.month); act["signup_month"] = pd.to_datetime(act.signup_month)

def observable(last_signup: pd.Series, tenure: pd.Series) -> pd.Series:
    """True where the latest signup in the cohort has reached this tenure by December 2025."""
    return (last_signup.dt.year * 12 + last_signup.dt.month + tenure) <= END.year * 12 + END.month

## 1. Logo retention triangle (signup quarter x months since signup)

In [2]:
cohort_size = act[act.tenure == 0].groupby("signup_quarter").customer_id.nunique().rename("customers")
last_q = act.groupby("signup_quarter").signup_month.max()
logo = act.groupby(["signup_quarter", "tenure"]).active.mean().mul(100).rename("logo_retention_pct").reset_index()
logo = logo[observable(logo.signup_quarter.map(last_q), logo.tenure)]
logo = logo.merge(cohort_size, left_on="signup_quarter", right_index=True)
save_table(logo, "4c_logo_retention_triangle")
tri = logo.pivot(index="signup_quarter", columns="tenure", values="logo_retention_pct")
tri.iloc[:, [0, 3, 6, 12, 18, 24, 36]].round(1).tail(16)

tenure,0,3,6,12,18,24,36
signup_quarter,,,,,,,
2022Q1,100.0,100.0,96.9,90.8,84.6,76.9,63.1
2022Q2,100.0,98.4,93.4,91.8,85.2,75.4,70.5
2022Q3,100.0,100.0,98.1,90.4,86.5,76.9,71.2
2022Q4,100.0,100.0,96.9,93.8,87.5,84.4,71.9
2023Q1,100.0,98.8,90.5,79.8,70.2,59.5,NaN
2023Q2,100.0,100.0,91.7,79.8,71.4,65.5,NaN
2023Q3,100.0,98.3,96.7,88.3,85.0,75.0,NaN
2023Q4,100.0,100.0,98.8,84.1,76.8,69.5,NaN
2024Q1,100.0,98.7,93.3,78.7,72.0,NaN,NaN


## 2. Revenue retention triangle (MRR relative to month 3, cohorts from 2022Q1)

Month 3 is the baseline because Analytics Add-on and Payments Module can be added in the first months after signup; measuring from month 0 would show the ramp-up as expansion. Figures are nominal and include the 2024 and 2025 price increases.

In [3]:
cm = sql_file(con, "cohort_mrr")
cm["month_start"] = pd.to_datetime(cm.month_start)
last_q_in = act[act.signup_month >= "2022-01-01"].groupby("signup_quarter").signup_month.max()
cm = cm[cm.signup_quarter.isin(last_q_in.index)]
cm = cm.groupby(["signup_quarter", "tenure"], as_index=False).mrr.sum()
base = cm[cm.tenure == 3].set_index("signup_quarter").mrr
cm = cm[cm.signup_quarter.isin(base.index) & (cm.tenure >= 3)]
cm = cm[observable(cm.signup_quarter.map(last_q_in), cm.tenure)]
cm["mrr_month3"] = cm.signup_quarter.map(base)
cm["revenue_retention_pct"] = 100 * cm.mrr / cm.mrr_month3
save_table(cm, "4c_revenue_retention_triangle")
rtri = cm.pivot(index="signup_quarter", columns="tenure", values="revenue_retention_pct")
rtri[[t for t in (3, 6, 9, 12, 18, 24, 36) if t in rtri.columns]].round(1)

tenure,3,6,9,12,18,24,36
signup_quarter,,,,,,,
2022Q1,100.0,96.0,92.7,93.9,89.5,86.9,77.1
2022Q2,100.0,96.1,96.2,97.8,94.6,92.1,91.8
2022Q3,100.0,100.6,97.5,95.6,97.8,87.4,90.6
2022Q4,100.0,98.9,99.6,97.7,94.9,92.3,89.0
2023Q1,100.0,92.5,88.0,89.2,84.9,77.2,NaN
2023Q2,100.0,92.6,95.6,89.6,77.7,77.9,NaN
2023Q3,100.0,103.8,100.9,99.8,102.6,92.6,NaN
2023Q4,100.0,100.6,95.8,93.0,93.5,87.3,NaN
2024Q1,100.0,99.3,86.8,92.3,86.1,NaN,NaN


## 3. Heatmaps

In [4]:
def heatmap(tri, title, fname, vmin, max_tenure, step):
    tri = tri.loc[:, [c for c in tri.columns if c <= max_tenure]]
    fig, ax = plt.subplots(figsize=(11, 0.22 * len(tri) + 1.6))
    im = ax.imshow(tri.values, aspect="auto", cmap=ch.SEQ_CMAP, vmin=vmin, vmax=100, interpolation="nearest")
    ax.set_yticks(range(len(tri))); ax.set_yticklabels(tri.index, fontsize=7)
    cols = list(tri.columns)
    ticks = [i for i, c in enumerate(cols) if c % step == 0]
    ax.set_xticks(ticks); ax.set_xticklabels([cols[i] for i in ticks], fontsize=7)
    ax.set_xlabel("Months since signup"); ax.set_ylabel("Signup quarter (cohort)")
    ax.grid(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
    cb.ax.yaxis.set_major_formatter(ch.pct_fmt()); cb.ax.tick_params(labelsize=7)
    ax.set_title(title)
    return ch.save(fig, fname)

print(heatmap(tri, "Logo retention by signup quarter: share of customers still subscribed (%)", "4c_logo_retention_heatmap", 50, 60, 6))
print(heatmap(rtri, "Revenue retention by signup quarter: MRR as % of month-3 MRR (cohorts from 2022Q1)", "4c_revenue_retention_heatmap", 50, 45, 3))

outputs/charts/4c_logo_retention_heatmap.png


outputs/charts/4c_revenue_retention_heatmap.png


## 4. Tenure-matched comparison: retention at 6, 12 and 18 months by signup year

Each figure is the share of a signup year's customers still subscribed after exactly 6, 12 or 18 months, so every cohort is measured over the same length of time. Years whose latest signups have not reached the tenure by December 2025 are left blank.

In [5]:
last_y = act.groupby("signup_year").signup_month.max()
rows = []
for T in (6, 12, 18):
    s = act[act.tenure == T].groupby("signup_year").active.agg(["mean", "size"])
    for y, r in s.iterrows():
        if observable(pd.Series([last_y[y]]), pd.Series([T])).iloc[0]:
            rows.append({"signup_year": y, "tenure_months": T, "logo_retention_pct": 100 * r["mean"], "customers": int(r["size"]),
                         "active_customers": int(round(r["mean"] * r["size"]))})
tm = pd.DataFrame(rows)
# revenue retention at the same tenures for in-window signup years
cmy = sql_file(con, "cohort_mrr").groupby(["signup_year", "tenure"], as_index=False).mrr.sum()
b3 = cmy[cmy.tenure == 3].set_index("signup_year").mrr
for T in (6, 12, 18):
    for y in b3.index:
        if observable(pd.Series([last_y[y]]), pd.Series([T])).iloc[0]:
            v = cmy[(cmy.signup_year == y) & (cmy.tenure == T)].mrr.sum()
            tm.loc[(tm.signup_year == y) & (tm.tenure_months == T), "revenue_retention_vs_m3_pct"] = 100 * v / b3[y]
tm = save_table(tm.sort_values(["tenure_months", "signup_year"]), "4c_tenure_matched")
tm.pivot(index="signup_year", columns="tenure_months", values="logo_retention_pct").round(1)

tenure_months,6,12,18
signup_year,,,
2017,97.7,93.6,88.9
2018,98.4,95.7,92.6
2019,97.1,92.2,87.8
2020,95.9,92.7,89.0
2021,97.0,90.2,86.0
2022,96.3,91.7,86.0
2023,94.2,82.6,75.2
2024,90.1,74.6,NaN


Grouped comparison: every customer who had reached the tenure by December 2025 is counted, so the 2023+ group at 18 months includes 2024 signups from the first half of the year. The year table above uses a stricter rule (the whole signup year must have reached the tenure).

In [6]:
# Older cohorts (2017 to 2022) against newer (2023 onwards) at the same tenure
tm["group"] = np.where(tm.signup_year <= 2022, "2017-2022 signups", "2023+ signups")
grp = act.assign(group=np.where(act.signup_year <= 2022, "2017-2022 signups", "2023+ signups"))
rows = []
for T in (6, 12, 18):
    for gname, gd in grp.groupby("group"):
        eligible = gd[observable(gd.signup_month, pd.Series(T, index=gd.index))]
        cell = eligible[eligible.tenure == T]
        rows.append({"group": gname, "tenure_months": T, "customers_reaching_tenure": len(cell),
                     "active_customers": int(cell.active.sum()), "logo_retention_pct": 100 * cell.active.mean()})
old_new = save_table(pd.DataFrame(rows), "4c_old_vs_new_cohorts")
old_new

,group,tenure_months,customers_reaching_tenure,active_customers,logo_retention_pct
0,2017-2022 signups,6,1259,1221,96.981732
1,2023+ signups,6,761,692,90.932983
2,2017-2022 signups,12,1259,1165,92.533757
3,2023+ signups,12,613,482,78.629690
4,2017-2022 signups,18,1259,1110,88.165210
5,2023+ signups,18,474,344,72.573840


In [7]:
# Why cumulative churn misleads: churn to date by signup year, which rises with age
cum = act[act.month == END].groupby("signup_year").active.agg(["mean", "size"]).reset_index()
cum["cumulative_churn_to_date_pct"] = 100 * (1 - cum["mean"])
cum["years_since_signup_approx"] = 2025 - cum.signup_year
cum = save_table(cum.rename(columns={"size": "customers"})[["signup_year", "customers", "years_since_signup_approx", "cumulative_churn_to_date_pct"]], "4c_cumulative_churn_by_year")
cum

,signup_year,customers,years_since_signup_approx,cumulative_churn_to_date_pct
0,2017,171,8,48.538012
1,2018,188,7,38.297872
2,2019,205,6,40.975610
3,2020,218,5,38.532110
4,2021,235,4,30.212766
5,2022,242,3,34.710744
6,2023,310,2,39.354839
7,2024,303,1,33.003300
8,2025,320,0,9.375000


In [8]:
fig, ax = plt.subplots(figsize=(8, 4.6))
p = tm.pivot(index="signup_year", columns="tenure_months", values="logo_retention_pct")
for T, colour in zip((6, 12, 18), ch.SERIES):
    s = p[T].dropna()
    ax.plot(s.index, s.values, color=colour, marker="o", ms=5, lw=2, label=f"after {T} months")
    if T == 18:  # label below the point so it does not meet the 12-month label
        ax.text(s.index[-1], s.values[-1] - 0.9, f"{T} months: {s.values[-1]:.1f}%", color=ch.TEXT, fontsize=8, ha="center", va="top")
    else:
        ax.text(s.index[-1] + 0.12, s.values[-1], f"{T} months: {s.values[-1]:.1f}%", color=ch.TEXT, fontsize=8, va="center")
ax.set_xticks(p.index); ax.set_xlim(p.index.min() - 0.3, p.index.max() + 1.3)
ax.yaxis.set_major_formatter(ch.pct_fmt())
ax.set_xlabel("Signup year (cohort)"); ax.set_ylabel("Customers still subscribed (%)")
ax.set_title("Logo retention at the same tenure, by signup year")
ax.legend(loc="lower left")
ax.set_ylim(p.min().min() - 3, 100)
print(ch.save(fig, "4c_tenure_matched"))

outputs/charts/4c_tenure_matched.png
